# FraudGraph — Stage 4: Explainability layer (v2)

Per-decision reasons for fraud ops: chargeback disputes, GDPR "right to
explanation", manual-review workflows.

**v2 fixes:**
- **Label-safe burst reasons.** v1 built the velocity-neighbor history from the
  full dataframe, so a test transaction's "burst: 1 already flagged fraud"
  could cite the label of a *test-period* neighbor — a label that in reality
  takes weeks to confirm. Now: neighbor *existence* comes from all history
  (transactions are observable facts the moment they happen), but fraud labels
  are only read for train-period neighbors (the confirmed-fraud ledger).
- **Faithfulness check strings updated** to match the current reason wording
  (v1 grepped for retired text and reported 0/0).
- **No-merchant rows handled**: Stage 2 v2 no longer assigns pseudo-merchants
  to missing-addr1 rows; those pass `merchant_idx=None`.

## 0. Setup

In [1]:
!pip install -q torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 752.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.0 MB/s eta 0:00:0000:01


In [2]:
import os, pickle, json
import numpy as np, pandas as pd, torch

OUT = "/kaggle/working"

## 1. Load graph_meta (Stage 2 v2 dataset input required)

In [3]:
import glob
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input"
    return hits[0]

META_PATH = find("graph_meta.pkl")
print("graph_meta:", META_PATH)
with open(META_PATH, "rb") as f:
    graph_meta = pickle.load(f)
assert graph_meta.get("merchant_requires_addr1"), \
    "old graph_meta.pkl — rerun Stage 2 v2 and update the Kaggle dataset first"

graph_meta: /kaggle/input/datasets/svkayy/fraudgraph-stage2/graph_meta.pkl


## 2. Reload raw data, rebuild entity indices

In [4]:
DATA = "/kaggle/input/competitions/ieee-fraud-detection"
txn = pd.read_csv(f"{DATA}/train_transaction.csv")
idn = pd.read_csv(f"{DATA}/train_identity.csv")
df = txn.merge(idn, on="TransactionID", how="left")
df = df.sort_values("TransactionDT").reset_index(drop=True)
cutoff = graph_meta["split_cutoff"]
is_train = df["TransactionDT"] <= cutoff

has_addr = df["addr1"].notna()
df["card1"] = df["card1"].fillna(-999)
df["addr1"] = df["addr1"].fillna(-999)
df["DeviceInfo_key"] = df["DeviceInfo"].fillna(graph_meta["device_unk"]).astype(str)
df["merchant_key"]   = df["ProductCD"].astype(str) + "|" + df["addr1"].astype(float).astype(str)

df["_card_idx"]  = df["card1"].astype(float).map(graph_meta["card_map"]).astype("int64")
df["_merch_idx"] = df["merchant_key"].map(graph_meta["merch_map"]).fillna(-1).astype("int64")
df.loc[~has_addr, "_merch_idx"] = -1
df["_dev_idx"]   = df["DeviceInfo_key"].map(graph_meta["dev_map"]).astype("int64")

train_df = df[is_train].reset_index(drop=True)
test_df  = df[~is_train].reset_index(drop=True)
print(f"train: {len(train_df):,}  test: {len(test_df):,}")
print(f"train fraud rate: {train_df['isFraud'].mean():.4f}")
print(f"rows without merchant: {(df['_merch_idx'] < 0).mean():.1%}")

train: 472,432  test: 118,108
train fraud rate: 0.0351
rows without merchant: 11.1%


## 3. Per-node fraud stats (train only — no leakage)

In [5]:
def agg_card(g):
    return pd.Series({
        "tx_count": len(g),
        "fraud_count": int(g["isFraud"].sum()),
        "fraud_rate": float(g["isFraud"].mean()),
        "avg_amount": float(g["TransactionAmt"].mean()),
        "unique_merchants": int(g["_merch_idx"].nunique()),
    })

def agg_merch(g):
    return pd.Series({
        "tx_count": len(g),
        "fraud_count": int(g["isFraud"].sum()),
        "fraud_rate": float(g["isFraud"].mean()),
    })

def agg_dev(g):
    return pd.Series({
        "tx_count": len(g),
        "fraud_count": int(g["isFraud"].sum()),
        "fraud_rate": float(g["isFraud"].mean()),
        "unique_cards": int(g["_card_idx"].nunique()),
    })

card_stats  = train_df.groupby("_card_idx").apply(agg_card, include_groups=False).to_dict(orient="index")
merch_stats = {k: v for k, v in
               train_df[train_df["_merch_idx"] >= 0].groupby("_merch_idx")
               .apply(agg_merch, include_groups=False).to_dict(orient="index").items()}
dev_stats   = train_df.groupby("_dev_idx").apply(agg_dev, include_groups=False).to_dict(orient="index")

print(f"cards: {len(card_stats):,}  merchants: {len(merch_stats):,}  devices: {len(dev_stats):,}")

cards: 12,730  merchants: 520  devices: 1,640


## 4. Velocity-cluster neighbors — label-safe

Neighbor **existence** uses all history (a transaction is an observable fact
the moment it happens). Neighbor **fraud labels** are only read from the
train period — the "confirmed fraud ledger" a production system would have.
Test-period neighbors appear with `is_fraud=None` (unknown yet).

In [6]:
WIN = graph_meta["velocity_window_sec"]

history = {}
for cid, t, fr, tr in zip(df["_card_idx"].astype(int), df["TransactionDT"].astype(float),
                          df["isFraud"].astype(int), is_train):
    history.setdefault(cid, []).append((t, bool(fr) if tr else None))
for k in history:
    history[k].sort(key=lambda x: x[0])

def velocity_neighbors_for(card_idx, tx_time):
    lst = history.get(int(card_idx), [])
    lo = tx_time - WIN
    return [{"is_fraud": f} for (t, f) in lst if lo <= t < tx_time]

sample = test_df.head(3)
for _, r in sample.iterrows():
    nb = velocity_neighbors_for(r["_card_idx"], r["TransactionDT"])
    known = sum(1 for n in nb if n["is_fraud"] is True)
    print(f"  tx={int(r['TransactionID'])}  cluster_size={len(nb)}  confirmed_fraud_in_cluster={known}")

  tx=3459432  cluster_size=0  confirmed_fraud_in_cluster=0
  tx=3459433  cluster_size=0  confirmed_fraud_in_cluster=0
  tx=3459434  cluster_size=0  confirmed_fraud_in_cluster=0


## 5. FraudExplainer (mirror `src/explainer.py`)

In [7]:
from dataclasses import dataclass, asdict

@dataclass
class Reason:
    factor: str
    weight: float
    edge_type: str
    evidence: dict

class FraudExplainer:
    RISKY_LIFT_MIN = 1.5
    PROTECTIVE_LIFT_MAX = 0.5
    MIN_HISTORY = 3

    def __init__(self, card_stats, merchant_stats, device_stats,
                 amt_mean, amt_std, train_fraud_rate, unknown_device_idx=None):
        self.card_stats = card_stats
        self.merchant_stats = merchant_stats
        self.device_stats = device_stats
        self.amt_mean = amt_mean
        self.amt_std = amt_std
        self.train_fraud_rate = train_fraud_rate
        self.unknown_device_idx = unknown_device_idx

    @staticmethod
    def _lift_weight(lift, base=0.1, scale=0.35, cap=1.0):
        return min(cap, base + scale * max(0.0, lift - 1.0))

    def explain(self, transaction_id, fraud_score, card_idx, merchant_idx,
                device_idx, amount, velocity_neighbors=None, top_k=3):
        reasons = []

        card = self.card_stats.get(card_idx) if card_idx is not None else None
        if card and card["tx_count"] >= self.MIN_HISTORY:
            lift = card["fraud_rate"] / max(self.train_fraud_rate, 1e-6)
            if lift >= self.RISKY_LIFT_MIN and card["fraud_count"] >= 2:
                reasons.append(Reason(
                    factor=(f"shared card with elevated fraud rate: "
                            f"{card['fraud_rate']*100:.1f}% "
                            f"({lift:.1f}x baseline, {int(card['fraud_count'])} prior fraud tx)"),
                    weight=self._lift_weight(lift),
                    edge_type="uses_card",
                    evidence={"card_idx": int(card_idx), **card, "lift": round(lift, 2)}))
            elif lift <= self.PROTECTIVE_LIFT_MAX and card["tx_count"] >= 10:
                reasons.append(Reason(
                    factor=(f"protective: card has below-baseline fraud rate "
                            f"({card['fraud_rate']*100:.1f}%, {lift:.2f}x baseline, "
                            f"{int(card['tx_count'])} prior tx)"),
                    weight=0.15, edge_type="uses_card",
                    evidence={"card_idx": int(card_idx), **card, "lift": round(lift, 2), "protective": True}))

        if device_idx is not None and device_idx != self.unknown_device_idx:
            dev = self.device_stats.get(device_idx)
            if dev and dev["tx_count"] >= self.MIN_HISTORY:
                lift = dev["fraud_rate"] / max(self.train_fraud_rate, 1e-6)
                if lift >= self.RISKY_LIFT_MIN and dev["fraud_count"] >= 2:
                    reasons.append(Reason(
                        factor=(f"device linked to {int(dev['unique_cards'])} cards, "
                                f"{int(dev['fraud_count'])} with prior fraud "
                                f"({lift:.1f}x baseline)"),
                        weight=self._lift_weight(lift),
                        edge_type="on_device",
                        evidence={"device_idx": int(device_idx), **dev, "lift": round(lift, 2)}))

        merch = self.merchant_stats.get(merchant_idx) if merchant_idx is not None else None
        if merch and merch["tx_count"] > 20:
            lift = merch["fraud_rate"] / max(self.train_fraud_rate, 1e-6)
            if lift >= self.RISKY_LIFT_MIN:
                reasons.append(Reason(
                    factor=(f"merchant has elevated fraud rate: "
                            f"{merch['fraud_rate']*100:.1f}% ({lift:.1f}x baseline)"),
                    weight=self._lift_weight(lift, scale=0.25, cap=0.7),
                    edge_type="at_merchant",
                    evidence={"merchant_idx": int(merchant_idx), **merch, "lift": round(lift, 2)}))

        if velocity_neighbors and len(velocity_neighbors) >= 2:
            n_fraud = sum(1 for n in velocity_neighbors if n.get("is_fraud") is True)
            if n_fraud > 0:
                reasons.append(Reason(
                    factor=(f"card-testing burst: {len(velocity_neighbors)} tx on this card in the "
                            f"last 5 minutes, {n_fraud} confirmed fraud"),
                    weight=min(1.0, 0.5 + 0.15 * n_fraud),
                    edge_type="velocity_cluster",
                    evidence={"cluster_size": len(velocity_neighbors), "fraud_in_cluster": n_fraud}))
            elif len(velocity_neighbors) >= 4:
                reasons.append(Reason(
                    factor=(f"card-testing burst: {len(velocity_neighbors)} tx on this card in the "
                            f"last 5 minutes"),
                    weight=0.3, edge_type="velocity_cluster",
                    evidence={"cluster_size": len(velocity_neighbors), "fraud_in_cluster": 0}))

        if self.amt_std > 0:
            z = (amount - self.amt_mean) / self.amt_std
            if abs(z) > 3.0:
                reasons.append(Reason(
                    factor=f"amount {abs(z):.1f}σ {'above' if z > 0 else 'below'} training-set mean",
                    weight=min(0.5, 0.05 + 0.05 * (abs(z) - 3)),
                    edge_type="self",
                    evidence={"amount": float(amount), "z_score": round(float(z), 2)}))

        if not reasons:
            reasons.append(Reason(
                factor=("no elevated-risk signals in card, merchant, or device history"
                        if fraud_score < 0.3 else
                        "model flagged based on feature combination without a single dominant relational cause"),
                weight=1.0, edge_type="none", evidence={}))

        reasons.sort(key=lambda r: r.weight, reverse=True)
        total = sum(r.weight for r in reasons) or 1.0
        for r in reasons:
            r.weight = round(r.weight / total, 3)

        return {
            "transaction_id": transaction_id,
            "fraud_score": round(float(fraud_score), 4),
            "top_reasons": [asdict(r) for r in reasons[:top_k]],
        }

train_fraud_rate = float(train_df["isFraud"].mean())
unknown_dev_idx = graph_meta["dev_map"].get(graph_meta["device_unk"])
print("unknown_device_idx (filtered):", unknown_dev_idx)
explainer = FraudExplainer(
    card_stats=card_stats,
    merchant_stats=merch_stats,
    device_stats=dev_stats,
    amt_mean=float(train_df["TransactionAmt"].mean()),
    amt_std=float(train_df["TransactionAmt"].std()),
    train_fraud_rate=train_fraud_rate,
    unknown_device_idx=unknown_dev_idx,
)
print("explainer built.  train_fraud_rate:", round(train_fraud_rate, 4))

unknown_device_idx (filtered): 1557
explainer built.  train_fraud_rate: 0.0351


## 6. Proxy score for demo purposes

The real GNN scores arrive in Stage 5; here we only evaluate *explanation
quality*, so a crude fraud-rate proxy suffices for picking demo samples.

In [8]:
def proxy_score(row):
    c = card_stats.get(int(row["_card_idx"]), {"fraud_rate": 0.0})
    d = dev_stats.get(int(row["_dev_idx"]), {"fraud_rate": 0.0})
    base = 0.5 * c["fraud_rate"] + 0.3 * d["fraud_rate"] + 0.2 * train_fraud_rate
    return float(np.clip(base * 3, 0.0, 1.0))

test_df["proxy_score"] = test_df.apply(proxy_score, axis=1)
print(test_df["proxy_score"].describe())

count    118108.000000
mean          0.099043
std           0.094908
min           0.021081
25%           0.049556
50%           0.070060
75%           0.102788
max           1.000000
Name: proxy_score, dtype: float64


## 7. Sample explanations (5 fraud + 5 legit)

In [9]:
fraud_samples = test_df[test_df["isFraud"] == 1].sample(5, random_state=0)
legit_samples = test_df[test_df["isFraud"] == 0].sample(5, random_state=0)

def show_row(row, label):
    print(f"\n{'='*78}\n{label}  txID={int(row['TransactionID'])}  isFraud={int(row['isFraud'])}\n{'-'*78}")
    nb = velocity_neighbors_for(row["_card_idx"], row["TransactionDT"])
    exp = explainer.explain(
        transaction_id=int(row["TransactionID"]),
        fraud_score=row["proxy_score"],
        card_idx=int(row["_card_idx"]),
        merchant_idx=int(row["_merch_idx"]) if row["_merch_idx"] >= 0 else None,
        device_idx=int(row["_dev_idx"]),
        amount=float(row["TransactionAmt"]),
        velocity_neighbors=nb,
    )
    print(f"score={exp['fraud_score']:.3f}")
    for i, r in enumerate(exp["top_reasons"], 1):
        print(f"  {i}. [{r['weight']:.2f}] {r['factor']}  ({r['edge_type']})")

for _, r in fraud_samples.iterrows():
    show_row(r, "FRAUD")
for _, r in legit_samples.iterrows():
    show_row(r, "LEGIT")


FRAUD  txID=3529178  isFraud=1
------------------------------------------------------------------------------
score=0.124
  1. [1.00] shared card with elevated fraud rate: 5.3% (1.5x baseline, 46 prior fraud tx)  (uses_card)

FRAUD  txID=3560515  isFraud=1
------------------------------------------------------------------------------
score=0.190
  1. [1.00] device linked to 15 cards, 5 with prior fraud (3.0x baseline)  (on_device)

FRAUD  txID=3530250  isFraud=1
------------------------------------------------------------------------------
score=0.097
  1. [1.00] no elevated-risk signals in card, merchant, or device history  (none)

FRAUD  txID=3553178  isFraud=1
------------------------------------------------------------------------------
score=0.118
  1. [1.00] no elevated-risk signals in card, merchant, or device history  (none)

FRAUD  txID=3475501  isFraud=1
------------------------------------------------------------------------------
score=0.082
  1. [1.00] no elevated-risk si

## 8. Coverage + faithfulness

Faithfulness is a **bug detector**: reasons are computed from the stats they're
checked against, so anything under 100% means the code lied. Coverage is the
honest quality number — what fraction of test-set fraud gets a substantive
(non-fallback) reason.

In [10]:
frauds = test_df[test_df["isFraud"] == 1]
covered, total = 0, 0
checks = {"uses_card_risky": [0, 0], "uses_card_protective": [0, 0],
          "velocity_cluster": [0, 0], "on_device": [0, 0]}

for _, r in frauds.iterrows():
    nb = velocity_neighbors_for(r["_card_idx"], r["TransactionDT"])
    exp = explainer.explain(
        transaction_id=int(r["TransactionID"]),
        fraud_score=r["proxy_score"],
        card_idx=int(r["_card_idx"]),
        merchant_idx=int(r["_merch_idx"]) if r["_merch_idx"] >= 0 else None,
        device_idx=int(r["_dev_idx"]),
        amount=float(r["TransactionAmt"]),
        velocity_neighbors=nb,
    )
    total += 1
    top_types = {rr["edge_type"] for rr in exp["top_reasons"]}
    if top_types - {"none"}:
        covered += 1

    cs = card_stats.get(int(r["_card_idx"]), {})
    ds = dev_stats.get(int(r["_dev_idx"]), {})
    for rr in exp["top_reasons"]:
        if rr["edge_type"] == "uses_card" and "elevated fraud rate" in rr["factor"]:
            checks["uses_card_risky"][1] += 1
            if cs.get("fraud_count", 0) >= 2:
                checks["uses_card_risky"][0] += 1
        if rr["edge_type"] == "uses_card" and "protective" in rr["factor"]:
            checks["uses_card_protective"][1] += 1
            if cs.get("tx_count", 0) >= 10:
                checks["uses_card_protective"][0] += 1
        if rr["edge_type"] == "velocity_cluster":
            checks["velocity_cluster"][1] += 1
            if len(nb) >= 2:
                checks["velocity_cluster"][0] += 1
        if rr["edge_type"] == "on_device":
            checks["on_device"][1] += 1
            if ds.get("fraud_count", 0) >= 2:
                checks["on_device"][0] += 1

print(f"Test-set fraud transactions: {total:,}")
print(f"Coverage (non-fallback reasons): {covered/total*100:.1f}%")
for name, (good, n) in checks.items():
    if n:
        print(f"Faithfulness — {name:22s}: {good}/{n} = {good/n*100:.1f}%")
    else:
        print(f"Faithfulness — {name:22s}: never fired on fraud sample")

Test-set fraud transactions: 4,064
Coverage (non-fallback reasons): 75.7%
Faithfulness — uses_card_risky       : 1881/1881 = 100.0%
Faithfulness — uses_card_protective  : 723/723 = 100.0%
Faithfulness — velocity_cluster      : 7/7 = 100.0%
Faithfulness — on_device             : 1168/1168 = 100.0%


## 9. Export for Stage 5 serving

In [11]:
def clean(d):
    return {int(k): {kk: float(vv) for kk, vv in v.items()} for k, v in d.items()}

# Train-only confirmed-fraud ledger for velocity lookups. In production this
# grows from the live stream + fraud-confirmation feedback loop.
card_history = {}
for cid, t, fr in zip(train_df["_card_idx"].astype(int),
                      train_df["TransactionDT"].astype(float),
                      train_df["isFraud"].astype(int)):
    card_history.setdefault(cid, []).append([t, fr])
for k in card_history:
    card_history[k].sort()

reference = {
    "card_stats":     clean(card_stats),
    "merchant_stats": clean(merch_stats),
    "device_stats":   clean(dev_stats),
    "card_map":     {float(k): int(v) for k, v in graph_meta["card_map"].items()},
    "merch_map":    {str(k): int(v) for k, v in graph_meta["merch_map"].items()},
    "dev_map":      {str(k): int(v) for k, v in graph_meta["dev_map"].items()},
    "device_unk":   graph_meta["device_unk"],
    "unknown_device_idx": unknown_dev_idx,
    "merchant_requires_addr1": True,
    "amt_mean":     float(train_df["TransactionAmt"].mean()),
    "amt_std":      float(train_df["TransactionAmt"].std()),
    "train_fraud_rate": train_fraud_rate,
    "velocity_window_sec": WIN,
    "split_cutoff": graph_meta["split_cutoff"],
    "card_history": card_history,
}

with open(f"{OUT}/explainer_reference.pkl", "wb") as f:
    pickle.dump(reference, f)

sz_mb = os.path.getsize(f"{OUT}/explainer_reference.pkl") / 1024**2
print(f"Saved: explainer_reference.pkl  ({sz_mb:.1f} MB)")

Saved: explainer_reference.pkl  (7.9 MB)
